**1. IMPORTS & SETUP**

In [ ]:
import os, random, warnings
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils import class_weight_
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

**2. DATASET AUTO-DETECTION (KAGGLE INPUT)**

In [ ]:
# -- 1. AUTO-DETECT DATASET PATH (FIX) -----------------------
import os

BASE_INPUT = "/kaggle/input"

print("Available datasets in /kaggle/input:")
print(os.listdir(BASE_INPUT))

DATASET_FOLDER = None

for root, dirs, files in os.walk(BASE_INPUT):
    if "Training" in dirs and "Testing" in dirs:
        DATASET_FOLDER = root
        break

if DATASET_FOLDER is None:
    raise FileNotFoundError("Could not auto-detect dataset folder")

print("Using dataset:", DATASET_FOLDER)

TRAIN_DIR = os.path.join(DATASET_FOLDER, "Training")
TEST_DIR  = os.path.join(DATASET_FOLDER, "Testing")

**3. CONFIGURATION**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

IMAGE_SIZE = 224
BATCH_SIZE = 64
CLASSES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASSES)

AUTOTUNE = tf.data.AUTOTUNE

**4. LOAD DATA**

In [ ]:
def load_data(directory):
    paths, labels = [], []
    for label in CLASSES:
        folder = os.path.join(directory, label)
        for fname in os.listdir(folder):
            if fname.endswith((".jpg", ".png", ".jpeg")):
                paths.append(os.path.join(folder, fname))
                labels.append(CLASSES.index(label))
    return np.array(paths), np.array(labels)

train_paths, train_labels = load_data(TRAIN_DIR)
test_paths, test_labels = load_data(TEST_DIR)

print("Train:", len(train_paths), "Test:", len(test_paths))

**5. TRAIN / VALIDATION SPLIT + CLASS WEIGHTS**

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(sss.split(train_paths, train_labels))

tr_paths, tr_labels = train_paths[train_idx], train_labels[train_idx]
val_paths, val_labels = train_paths[val_idx], train_labels[val_idx]

cw = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(tr_labels),
    y=tr_labels
)
class_weights = dict(enumerate(cw))

**6. DATA PIPELINE (PREPROCESS + AUGMENT)**

In [ ]:
def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.vgg16.preprocess_input(img)
    return img, label

def augment(path, label):
    img, label = preprocess(path, label)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, dtype=tf.int32))
    return img, label

def make_ds(paths, labels, aug=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if aug:
        ds = ds.shuffle(len(paths))
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_ds(tr_paths, tr_labels, aug=True)
val_ds = make_ds(val_paths, val_labels)
test_ds = make_ds(test_paths, test_labels)

**7. MODEL (VGG16 TRANSFER LEARNING)**

In [ ]:
base = VGG16(include_top=False, weights='imagenet',
             input_shape=(224, 224, 3))
base.trainable = False

inputs = Input(shape=(224, 224, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

**8. TRAINING PHASE 1 (FROZEN BASE)**

In [ ]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights
)

**9. FINE TUNING (UNFREEZE TOP LAYERS)**

In [ ]:
base.trainable = True
for layer in base.layers:
    layer.trainable = layer.name.startswith("block5")

model.compile(
    optimizer=Adam(5e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weights
)

**10. EVALUATION**

In [ ]:
pred = np.argmax(model.predict(test_ds), axis=1)

print(classification_report(test_labels, pred, target_names=CLASSES))

cm = confusion_matrix(test_labels, pred)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.show()

**11. Save Model**

In [2]:
SAVE_PATH = "brain_tumor_model.h5"

# Save model
model.save(SAVE_PATH)
print("Model saved:", SAVE_PATH)

# Reload (sanity check like Colab)
reloaded = tf.keras.models.load_model(SAVE_PATH, compile=False)

print("Reloaded model.input  :", reloaded.input)
print("Reloaded model.output :", reloaded.output)
print("✅ Reload verified — Grad-CAM will work in Flask")

# Download (Kaggle version of files.download)
from IPython.display import FileLink

FileLink(SAVE_PATH)

Model saved: brain_tumor_model.h5
Reloaded model.input  : [<KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=input_layer_1>]
Reloaded model.output : <KerasTensor shape=(None, 4), dtype=float32, sparse=False, ragged=False, name=keras_tensor_73>
✅ Reload verified — Grad-CAM will work in Flask


/kaggle/working/brain_tumor_model.h5